# 05 — Live USGS earthquakes → H3 → anomaly detection

**[🚀 Launch this notebook live](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/lite/lab/index.html?path=v2_05_h3_earthquake_geoai.ipynb)**

This notebook is intentionally dynamic: it reads the USGS **all earthquakes, past 30 days** GeoJSON feed, aggregates events into H3 cells, then uses an Isolation Forest to highlight cells with unusual combinations of event count and magnitude.

Because the source feed changes continuously, record the run timestamp when using the result outside this tutorial.

> **JupyterLite compatibility:** this notebook uses `geolibre_lite.LiteMap`, which builds native GeoLibre project/layer definitions and displays them through GeoLibre's hosted `?embed=1` viewer. It deliberately avoids `geolibre.Map`, whose localhost HTTP server cannot bind a socket inside Pyodide/WebAssembly.


In [ ]:
import sys
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["geolibre==3.0.0", "pyodide-http"])
from geolibre_lite import LiteMap as Map


In [ ]:
import sys, requests, pandas as pd, numpy as np
if sys.platform == "emscripten":
    import pyodide_http
    pyodide_http.patch_all()

import h3
from sklearn.ensemble import IsolationForest


In [ ]:
USGS = "https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_month.geojson"
response = requests.get(USGS, timeout=60)
response.raise_for_status()
feed = response.json()

rows = []
for f in feed["features"]:
    lon, lat, depth = f["geometry"]["coordinates"][:3]
    p = f["properties"]
    rows.append({
        "lon": lon, "lat": lat, "depth_km": depth,
        "mag": p.get("mag") if p.get("mag") is not None else 0.0,
        "place": p.get("place"), "time": p.get("time"),
    })

eq = pd.DataFrame(rows)
eq["h3"] = [h3.latlng_to_cell(lat, lon, 3) for lat, lon in zip(eq.lat, eq.lon)]
eq.head(), len(eq)


In [ ]:
agg = (
    eq.groupby("h3")
      .agg(count=("mag", "size"), max_mag=("mag", "max"), mean_mag=("mag", "mean"), mean_depth=("depth_km", "mean"))
      .reset_index()
)

X = agg[["count", "max_mag", "mean_mag", "mean_depth"]].fillna(0)
iso = IsolationForest(n_estimators=200, contamination="auto", random_state=42)
iso.fit(X)
agg["anomaly_score"] = -iso.score_samples(X)  # larger = more unusual
agg.sort_values("anomaly_score", ascending=False).head(10)


In [ ]:
features = []
for row in agg.itertuples(index=False):
    boundary = h3.cell_to_boundary(row.h3)
    ring = [[lng, lat] for lat, lng in boundary]
    ring.append(ring[0])
    features.append({
        "type": "Feature",
        "properties": {
            "h3": row.h3,
            "count": int(row.count),
            "max_mag": float(row.max_mag),
            "mean_mag": float(row.mean_mag),
            "mean_depth": float(row.mean_depth),
            "anomaly_score": float(row.anomaly_score),
        },
        "geometry": {"type": "Polygon", "coordinates": [ring]},
    })

h3_fc = {"type": "FeatureCollection", "features": features}


In [ ]:
m = Map(center=(-160, 15), zoom=2, height="720px")
m.add_choropleth(
    h3_fc,
    column="anomaly_score",
    name="H3 anomaly score",
    class_count=7,
    colormap="turbo",
    scheme="quantile",
    fillOpacity=0.55,
)
m.add_heatmap(
    eq.to_dict("records"),
    name="Earthquake density",
    radius=22,
    intensity=0.8,
    weight_field="mag",
)
m


## Model caveat

The Isolation Forest identifies statistical outliers in the chosen features. It does **not** predict earthquakes, estimate seismic hazard, or replace USGS hazard products. This is a visualization/feature-engineering demonstration.

## Data & software citations

- USGS Earthquake Hazards Program GeoJSON feeds: https://earthquake.usgs.gov/earthquakes/feed/v1.0/geojson.php
- Live feed: `https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_month.geojson`
- H3: https://h3geo.org/ (Apache-2.0).
- scikit-learn Isolation Forest: https://scikit-learn.org/
- GeoLibre: https://geolibre.app/
